# Executing parameterized quantum programs

Bind and run a single native circuit template at the original sweep angles. Measure finite counts, recover actual local jobs, and optimize the two-angle Bell circuit.

## Runtime circuit binding and qfunc specialization are different

This tutorial deliberately uses `fq.Circuit` for a symbolic angle sweep: one
compiled template retains a parameter and is bound repeatedly. This is a native
FlagQuantum capability, but it is not runtime parameter binding for synth qfunc.
The visible supplementary qfunc below shows the two-angle Bell preparation used
by the optimization. Its real array is bound at synthesis time, so every new
binding produces a specialized program.

The first template uses X, RY and CX. `workflow.template()` keeps that three-gate
construction in a reusable Python function; its implementation is in
[workflow.py](workflow.py). The later two-angle optimization is evaluated with
actual runs and a 200-query ceiling, not an analytic solution substituted into
the objective. Local archived jobs are local results, not cloud job execution.

Try a different finite pair of angles in the qfunc binding below and compare its
named probabilities. Keep symbolic Circuit binding when studying the original
50-angle reuse path. Unifying these binding behaviors is a compiler/API task,
not something this Notebook silently emulates.


Construct the original symbolic two-qubit circuit once.

In [ ]:
import math
from pathlib import Path

from IPython.display import Image, display

import examples.synthesis.tutorials.basics.quantum_programming.execution_tutorial_part2.overview.jobs as jobs
import examples.synthesis.tutorials.basics.quantum_programming.execution_tutorial_part2.overview.main as report
import examples.synthesis.tutorials.basics.quantum_programming.execution_tutorial_part2.overview.workflow as workflow

template = workflow.template()
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import CX, RX, Output, QBit, R, allocate, qfunc

@qfunc
def entangled_angles(angles: list[float], x: Output[QBit], y: Output[QBit]) -> None:
    allocate(x)
    allocate(y)
    RX(angles[0], x)
    R(angles[1], math.pi / 2, x)
    CX(x, y)

native_program = synth.synthesize(
    entangled_angles, classical_inputs={"angles": [0.37, -0.81]},
    max_qubits=2, max_workspace=0, memory_limit_bytes=1024**2,
)
print("Specialized qfunc probabilities:", dict(native_program.probabilities(fq.run(native_program), registers=("x", "y"))))


Choose the original pi-over-two value.

In [ ]:
angle = math.pi / 2

Draw the original single-binding counts from the bound circuit.

In [ ]:
assert angle == math.pi / 2
report.report_single_counts(template)

Reuse the symbolic template for all 50 angles.

In [ ]:
rows = workflow.sweep(template)

Read the two outcome populations from every finite record.

In [ ]:
report.report_sweep(rows)

Plot actual counts for the complete input sweep.

In [ ]:
figures = Path("notebook-figures")
report.plot(rows, figures)
display(Image(filename=str(figures / "angle-counts.png")))

Use the four-term Bell projector in the local observable.

In [ ]:
projector = workflow.PROJECTOR

Measure the projector at the original pi-over-five input.

In [ ]:
report.report_single_expectation(template)

Use both exact and finite measured expectations for every angle.

In [ ]:
assert all(row.projector_shots == 2048 for row in rows)

Plot the expectation and finite sampling outcomes.

In [ ]:
display(Image(filename=str(figures / "projector-sweep.png")))

Execute both request kinds with the local result store.

In [ ]:
archive = Path("notebook-jobs")
sample_id, observe_id = jobs.submit_both(template, math.pi / 5, archive)

Read the actual returned sample and projector by local content ID.

In [ ]:
report.report_jobs(sample_id, observe_id, archive)

Construct the original two-angle Bell preparation circuit.

In [ ]:
bell = workflow.bell_template()
assert bell.parameter_names == ("rx", "ry")

Minimize the negative Bell projector.

In [ ]:
cost = -1 * projector

Run the original 200-query variational budget from zero angles.

In [ ]:
optimization = workflow.optimize()

Read the last actual and best evaluated fidelities and coefficients.

In [ ]:
report.report_optimizer(optimization)